# AI Audio Detector — Google Colab Training

**Before running:**
1. Go to **Runtime → Change runtime type → GPU** (T4 is free, A100 is faster)
2. Make sure your Google Drive has this folder structure:
   ```
   My Drive/
   └── AI-Innovation-Data/
       ├── raw/
       │   ├── voice/real/       ← Anh's ASVspoof + WaveFake bonafide files
       │   ├── voice/fake/       ← Anh's ASVspoof + WaveFake spoof files
       │   ├── music/real/       ← MusicCaps (auto-downloaded) or manual
       │   ├── music/fake/       ← Anh's AI music (Suno/Udio)
       │   ├── non_human/real/   ← ESC-50 ZIP extracted here
       │   └── non_human/fake/   ← ElevenLabs SFX / LAION-AI
       └── checkpoints/          ← training saves here automatically
   ```
3. Run cells top to bottom. **Edit Cell 4** to set your Drive paths.


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 1 — Mount Google Drive
# ─────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('Drive mounted at /content/drive')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 2 — Clone the repository
# ─────────────────────────────────────────────────────────────────
import os

REPO_DIR = '/content/AI-Innovation'
BRANCH   = 'AI_sounds'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} https://github.com/foojanbabaeeian/AI-Innovation.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
print('Working directory:', os.getcwd())

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 3 — Install dependencies
# ─────────────────────────────────────────────────────────────────
!pip install -q -r requirements.txt
!apt-get install -q -y ffmpeg libsndfile1

# Verify GPU
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 4 — SET YOUR PATHS HERE (edit this cell)
# ─────────────────────────────────────────────────────────────────
import os, shutil
from pathlib import Path

# Root of your AI-Innovation-Data folder on Google Drive
DRIVE_DATA_ROOT = '/content/drive/MyDrive/AI-Innovation-Data'

# Where to save checkpoints (inside Drive so they persist after Colab disconnects)
DRIVE_CHECKPOINT_DIR = f'{DRIVE_DATA_ROOT}/checkpoints'

# Create symlinks from repo data/ → Drive so ingestors and the manifest work
RAW_LINK   = 'data/raw'
CKPT_LINK  = 'outputs'

os.makedirs('data/metadata', exist_ok=True)
Path(DRIVE_CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

for link, target in [(RAW_LINK, f'{DRIVE_DATA_ROOT}/raw'), (CKPT_LINK, DRIVE_CHECKPOINT_DIR)]:
    if os.path.islink(link):
        os.unlink(link)
    elif os.path.exists(link):
        shutil.rmtree(link)
    os.symlink(target, link)
    print(f'Linked {link} → {target}')

!ls data/raw/

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 5 — Build the master manifest
#          (scans data/raw/ and writes data/metadata/master_manifest.csv)
# ─────────────────────────────────────────────────────────────────
from src.data.preprocessing import run_preprocessing

n_rows = run_preprocessing(
    raw_dir='data/raw',
    output_dir='data/processed',
    manifest_path='data/metadata/master_manifest.csv',
    num_workers=4,
)
print(f'Manifest: {n_rows} rows')

# Show class balance
import pandas as pd
df = pd.read_csv('data/metadata/master_manifest.csv')
print(df.groupby(['domain', 'split', 'label']).size().unstack(fill_value=0))

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 6 — Smoke-test the data pipeline (loads one batch)
# ─────────────────────────────────────────────────────────────────
from src.data.dataset import build_dataloader

loader = build_dataloader(
    manifest_path='data/metadata/master_manifest.csv',
    data_root='data',
    split='train',
    batch_size=4,
    num_workers=2,
    max_samples=20,
)
batch = next(iter(loader))
print('waveform shape:', batch['waveform'].shape)     # (4, 1, 64000)
print('ai_ratio:', batch['ai_ratio'])
print('class_label:', batch['class_label'])

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 7 — Train the model
#
# GPU config: batch_size=64, gradient_accumulation_steps=1
# CPU config: batch_size=32, gradient_accumulation_steps=4 (default)
# ─────────────────────────────────────────────────────────────────
import subprocess, sys

# Override batch size for GPU
GPU_OVERRIDES = ''
if torch.cuda.is_available():
    GPU_OVERRIDES = '--batch_size 64 --gradient_accumulation_steps 1 --num_workers 4'

cmd = f'python -m src.training.train --config configs/default.yaml {GPU_OVERRIDES}'
print('Running:', cmd)
!{cmd}

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 8 — Export best checkpoint to ONNX
#          (for optional client-side inference)
# ─────────────────────────────────────────────────────────────────
import os

best_ckpt = 'outputs/ai_audio_detection/checkpoint_best.pt'
onnx_out  = f'{DRIVE_CHECKPOINT_DIR}/model.onnx'

if os.path.exists(best_ckpt):
    !python src/inference/export_onnx.py --checkpoint {best_ckpt} --output {onnx_out}
    print('ONNX model saved to Drive:', onnx_out)
else:
    print('No checkpoint found yet — run Cell 7 first')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 9 — Download best checkpoint to your local machine
#          (skip if you will use the model in Colab directly)
# ─────────────────────────────────────────────────────────────────
from google.colab import files

best_ckpt = 'outputs/ai_audio_detection/checkpoint_best.pt'
if os.path.exists(best_ckpt):
    files.download(best_ckpt)
else:
    print('No checkpoint yet')